# A_07 One shared model file

*Course appendix A: the toolbox (our own, not the book's appendix A). Dip in whenever you meet a tool you don't know yet.*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, write down what you think it will do. Then run it and compare.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll see `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → [model learns] → model writes → MCP tool
> ```
>
> You are here: **model learns**
>
> Project files this connects to:
>
> - `src/homer_ai/train_homer.py`
> - `src/homer_ai/model.py`
> - `src/homer_ai/homer_mcp_server.py`
>
> **This notebook:** Two files need the model class. Why does it live in one file, `model.py`, that both of them import, instead of being copied into each?
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- Why one copy of the model lives in `model.py`, and both other files import it
- Which of our files use which
- What goes wrong with two copies of a model: a saved file that won't load

## 2. Words you'll meet

No new words this time.

## 3. The idea

### ❓ Why is the model in `model.py`, and not inside each file that uses it?

Two jobs need the model class, `HomerGreekLLM`: training a new model (`train_homer.py`), and loading a saved one back so the MCP server can use it. The quick way is to copy the class into both files. We keep one copy and import it.

```
THE TRAP: two copies                WHAT WE DO: one blueprint

train_homer.py                      model.py
   class HomerGreekLLM  (copy 1)       class HomerGreekLLM  (the only one)
   ...training...                        ▲            ▲
                                  import │            │ import
homer_mcp_server.py                train_homer.py   checkpoint.py  ◄── homer_mcp_server.py
   class HomerGreekLLM  (copy 2)                    (loads a saved model)
```

The danger is specific to AI work. A saved model is just numbers, and they only make sense for the exact shape that saved them. Change one copy and forget the other, and the saved model **won't load**. So the class lives in `model.py` and nowhere else. The training script imports it, and so does `checkpoint.py`, the one place that turns a saved file back into a model. The server asks `checkpoint.py` and never names the class itself. One blueprint, used everywhere.

*Coming from C#, this is the first rule I'd set in any codebase: one definition, imported everywhere, never copied.*

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ Which of our files use which?

This code reads our own source files and prints the lines that start with `from homer_ai`. It's a live map of who uses whom. (We read the files as text, so there's no need to import the MCP server.)

👉 **Predict first:** which files import `model.py`?

In [ ]:
from pathlib import Path

import homer_ai

folder = Path(homer_ai.__file__).parent
for file in sorted(folder.glob("*.py")):
    lines = file.read_text(encoding="utf-8").splitlines()
    imports = [line for line in lines if line.startswith("from homer_ai")]
    print(file.name, "imports:", imports if imports else "nothing from homer_ai")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Three files import `model.py`: `train_homer.py`, `checkpoint.py` and `lessons.py` (the notebooks' helpers). `homer_mcp_server.py` isn't one of them. It imports `checkpoint.py`, and `checkpoint.py` imports the two model files, `model.py` and `gpt.py`. At the top, `model.py`, `gpt.py` and `tokenizer.py` import nothing from the project. The main lines, as a picture:

```
model.py    gpt.py    tokenizer.py       import nothing from our project
    ▲          ▲           ▲
    └──────────┼───────────┘
         checkpoint.py                   saves a model, and builds one from a saved file
           ▲         ▲
           │         │
train_homer.py     homer_mcp_server.py
```

An arrow points from the file that imports to the file it uses. (`train_homer.py` also imports the three top files directly, and the server imports `tokenizer.py` and `generation.py`.)

</details>

### ❓ What really goes wrong with two copies?

Say you change `embed_dim` (the size of each character's list of numbers) in one copy and forget the other. The saved weights then don't fit the other model.

👉 **Predict first:** a model with `embed_dim=32` saves its weights. Can a model with `embed_dim=16` load them?

In [ ]:
from homer_ai.model import HomerGreekLLM

saved = HomerGreekLLM(vocab_size=10, embed_dim=32).state_dict()  # like homer_model.pt
forgotten_copy = HomerGreekLLM(vocab_size=10, embed_dim=16)  # a copy nobody updated

try:
    forgotten_copy.load_state_dict(saved)
except RuntimeError as error:
    print("Could not load:", str(error).splitlines()[1].strip())

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

You get an error about a size mismatch. The saved weights are shaped for 32, the copy for 16. With one `model.py`, there's only one place to change, so this can't happen.

</details>

## 5. Recap

- Put the model in one file and import it everywhere, so copies can't drift apart.
- A saved model only fits the exact shape that saved it, so two copies that drift apart break loading.

**Next:** back to the course: [the course map](../00_start_here/00_00_course_map.ipynb).